<a href="https://colab.research.google.com/github/Maryy-666/Maryy-666/blob/main/Copy_of_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow==1.1.48 --quiet

import roboflow

roboflow.login()

rf = roboflow.Roboflow()

project = rf.workspace("model-examples").project("football-players-obj-detection")
dataset = project.version(2).download("yolov8")

/content/{HOME}/datasets
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 12.6 MB/s eta 0:00:00
visit https://app.roboflow.com/auth-cli to get your authentication token.


In [ ]:
!pip install ultralytics

!yolo task=detect \
      mode=train \
      model=yolov8n.pt \
      data={dataset.location}/data.yaml \
      epochs=25 \
      imgsz=640

### Load the trained model and perform inference

After training, you can load your best-performing model (`best.pt`) and use it for inference on new images. We'll pick an image from the validation set to demonstrate this.

In [ ]:
from ultralytics import YOLO
import os

# Load the trained model
model_path = os.path.join(os.getcwd(), 'runs/detect/train/weights/best.pt')
model = YOLO(model_path)

print(f"Model loaded from: {model_path}")

Now, let's run the model on an example image from the validation set. We'll get a list of validation images from the `data.yaml` file that was downloaded with your dataset.

In [ ]:
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Path to the data.yaml file
data_yaml_path = os.path.join(dataset.location, 'data.yaml')

# Read the data.yaml file
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Get the path to the validation images
val_images_path = data_config['val']

# List all image files in the validation directory
val_image_files = [os.path.join(val_images_path, f) for f in os.listdir(val_images_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if val_image_files:
    # Pick a random image from the validation set
    example_image_path = random.choice(val_image_files)
    print(f"Running inference on: {example_image_path}")

    # Run inference
    results = model.predict(example_image_path, save=True, conf=0.5)

    # Display results (YOLOv8 automatically saves annotated images to 'runs/detect/predict')
    # The path to the saved image will be in results[0].path if available.
    # However, it's easier to just display the image directly or show the saved one.

    # Find the path to the saved annotated image
    # The 'save=True' argument saves the image to a directory like 'runs/detect/predict'
    # The first result object contains information about the prediction
    if results and results[0].path:
        # Construct the path to the annotated image, which is usually in the 'predict' folder
        # after the current working directory
        output_dir = os.path.join(os.getcwd(), 'runs/detect/predict')
        annotated_image_name = os.path.basename(results[0].path)
        annotated_image_path = os.path.join(output_dir, annotated_image_name)

        if os.path.exists(annotated_image_path):
            print(f"Annotated image saved at: {annotated_image_path}")
            img = mpimg.imread(annotated_image_path)
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.axis('off')
            plt.title('Inference Result')
            plt.show()
        else:
            print(f"Could not find annotated image at {annotated_image_path}.")
    else:
        print("No results or annotated image path found.")
else:
    print("No validation images found to run inference on.")

### Export the trained model

You can export the trained YOLOv8 model to various formats suitable for deployment, such as ONNX or TensorFlow Lite (TFLite). This allows you to use the model with different inference engines or on specific hardware.

In [ ]:
print("Exporting model to ONNX format...")
# Ensure 'model' is defined. It should have been defined in a preceding cell.
# If for any reason it's not, reload it to prevent errors.
if 'model' not in locals():
    print("Warning: 'model' not found. Re-loading the trained model.")
    from ultralytics import YOLO
    import os
    model_path = os.path.join(os.getcwd(), 'runs/detect/train/weights/best.pt')
    model = YOLO(model_path)

# Export the model to ONNX format
model.export(format='onnx')
print("Model exported to ONNX format. You can find it in runs/detect/train/weights/best.onnx")

In [ ]:
print("Exporting model to TFLite format...")
# Ensure 'model' is defined. It should have been defined in a preceding cell.
# If for any reason it's not, reload it to prevent errors.
if 'model' not in locals():
    print("Warning: 'model' not found. Re-loading the trained model.")
    from ultralytics import YOLO
    import os
    model_path = os.path.join(os.getcwd(), 'runs/detect/train/weights/best.pt')
    model = YOLO(model_path)

# Export the model to TFLite format
# Note: TFLite export might require additional dependencies or specific configurations depending on the model architecture and desired quantization.
try:
    model.export(format='tflite')
    print("Model exported to TFLite format. You can find it in runs/detect/train/weights/best.tflite")
except Exception as e:
    print(f"Could not export to TFLite. Error: {e}")
    print("TFLite export often requires specific environment setups or might not be supported for all model configurations directly.")